<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M2_ENC_v1_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== M2_ENC_v1 Cell 1: 설정 + FULL_CONVERT 게이트 + 데이터셋 구조 확인 (읽기 전용) =====
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, time, hashlib, datetime
from pathlib import Path
import numpy as np, pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb")
DATASET_DIR  = PROJECT_ROOT / "bridge_datasets" / "FEMTO_2048_v1_2"
OUT_ROOT     = PROJECT_ROOT / "bridge_outputs"
CONTRACT_VERSION = "M2_ENC_CONTRACT_v1"
FS, NS, REG_MIN = 20480, 2048, 90
EXPECTED = {"LEARNING": (7534, 6), "TEST": (14127, 11), "FULL_TEST": (17355, 11)}
BASELINE_D025 = {"detected": 6, "mean_lead_h": 0.8356, "mean_reg_far_pct": 0.424}
CRITERIA_D028 = {"detected_min": 6, "mean_lead_h_min": 0.752, "mean_reg_far_pct_max": 0.924}
FOLDS = {"Bearing1_3":0,"Bearing2_3":0,"Bearing1_4":1,"Bearing2_4":1,
         "Bearing1_5":2,"Bearing2_5":2,"Bearing3_3":2,
         "Bearing1_6":3,"Bearing2_6":3,"Bearing1_7":4,"Bearing2_7":4}

def stop(code, **kw):
    raise RuntimeError(f"{code} | {json.dumps(kw, ensure_ascii=False, default=str)[:3000]}")

if not DATASET_DIR.is_dir():
    stop("M2ENC_BLOCKED_DATASET_NOT_FOUND", path=str(DATASET_DIR))

# --- 게이트: FULL_CONVERT_PASS 문자열이 들어 있는 요약 JSON 탐색 ---
def _json_has(o, tok):
    if isinstance(o, dict): return any(_json_has(v, tok) for v in o.values())
    if isinstance(o, list): return any(_json_has(v, tok) for v in o)
    return isinstance(o, str) and tok in o

cands = list(DATASET_DIR.glob("*.json"))
for d in OUT_ROOT.iterdir():
    if d.is_dir() and "full_convert" in d.name.lower():
        cands += list(d.glob("*.json"))
GATE_FILES = []
for p in sorted(set(cands)):
    try:
        if _json_has(json.loads(p.read_text(encoding="utf-8-sig")), "FULL_CONVERT_PASS"):
            GATE_FILES.append(p)
    except Exception as e:
        print("[GATE] 읽기 실패:", p.name, type(e).__name__)
if not GATE_FILES:
    stop("M2ENC_BLOCKED_FULL_CONVERT_GATE", searched=[str(p) for p in cands][:30])
print("[GATE] FULL_CONVERT_PASS 확인:", GATE_FILES[0])

# --- 구조 확인 ---
top = sorted(DATASET_DIR.iterdir())
print(f"\n[DATASET] 최상위 항목 {len(top)}개 (최대 40개 표시)")
for p in top[:40]:
    print("  ", "D" if p.is_dir() else "F", p.name)

INDEX_FILES = sorted(DATASET_DIR.rglob("index.csv"))
NPY_FILES   = sorted(DATASET_DIR.rglob("*.npy"))
print(f"\n[INDEX] index.csv {len(INDEX_FILES)}개")
for p in INDEX_FILES[:5]: print("   ", p.relative_to(DATASET_DIR))
print(f"[NPY] .npy {len(NPY_FILES)}개")
for p in NPY_FILES[:5]: print("   ", p.relative_to(DATASET_DIR))

if INDEX_FILES:
    _df = pd.read_csv(INDEX_FILES[0], nrows=5)
    print("\n[INDEX 열 목록]", list(_df.columns))
    print(_df.head(3).T)
print("\nCell 1 완료")


Mounted at /content/drive
[GATE] FULL_CONVERT_PASS 확인: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20261024_001446_full_convert_v1_2/FULL_CONVERT_FINAL_SUMMARY.json

[DATASET] 최상위 항목 5개 (최대 40개 표시)
   D FULL_TEST
   D LEARNING
   D TEST
   F dataset_contract.json
   F dataset_manifest.csv

[INDEX] index.csv 28개
    FULL_TEST/Bearing1_3/index.csv
    FULL_TEST/Bearing1_4/index.csv
    FULL_TEST/Bearing1_5/index.csv
    FULL_TEST/Bearing1_6/index.csv
    FULL_TEST/Bearing1_7/index.csv
[NPY] .npy 28개
    FULL_TEST/Bearing1_3/signals.npy
    FULL_TEST/Bearing1_4/signals.npy
    FULL_TEST/Bearing1_5/signals.npy
    FULL_TEST/Bearing1_6/signals.npy
    FULL_TEST/Bearing1_7/signals.npy

[INDEX 열 목록] ['row_index', 'split', 'record_key', 'logical_bearing_id', 'sequence_index', 'csv_file_name', 'source_absolute_path', 'source_file_size_bytes', 'source_modified_time_ns', 'source_quick_fingerprint', 'source_delimiter', 'source_sha256_at_read', 'output_array_relative_path', 

In [ ]:
# ===== M2_ENC_v1 Cell 2: 레코드 표 + 분할(D-026) + fold 고정 =====
def pick(df, cands, what, required=True):
    for c in cands:
        if c in df.columns: return c
    if required:
        stop("M2ENC_COLUMN_NOT_FOUND", what=what, candidates=cands, available=list(df.columns))
    return None

def norm_split(v):
    s = str(v).strip().upper().replace("-", "_").replace(" ", "_")
    return {"FULLTEST": "FULL_TEST", "FULL_TEST_SET": "FULL_TEST",
            "TEST_SET": "TEST", "LEARNING_SET": "LEARNING"}.get(s, s)

def norm_bearing(v):
    m = re.search(r"(\d)_(\d)", str(v))
    return f"Bearing{m.group(1)}_{m.group(2)}" if m else str(v).strip()

# index 읽기 (전역 1개 또는 레코드별 여러 개)
parts = []
for p in INDEX_FILES:
    d = pd.read_csv(p)
    d["_index_file"] = str(p)
    parts.append(d)
if not parts: stop("M2ENC_INDEX_NOT_FOUND", dataset=str(DATASET_DIR))
IDX = pd.concat(parts, ignore_index=True)

c_split = pick(IDX, ["split_normalized","split","_split","source_split"], "split")
c_bear  = pick(IDX, ["logical_bearing_id_n","logical_bearing_id","bearing_id","bearing","_bearing"], "bearing")
c_seq   = pick(IDX, ["sequence_index","source_sequence_index","_sequence","seq"], "sequence")
c_npy   = pick(IDX, ["signals_path","npy_path","record_npy_path","output_npy","output_path","signals_file"], "npy", required=False)
c_row   = pick(IDX, ["row_in_record","npy_row","record_row","row_in_npy","local_row","row_in_signals"], "row", required=False)

IDX["_split"]   = IDX[c_split].map(norm_split)
IDX["_bearing"] = IDX[c_bear].map(norm_bearing)
IDX["_seq"]     = pd.to_numeric(IDX[c_seq], errors="coerce")
if IDX["_seq"].isna().any(): stop("M2ENC_INVALID_SEQUENCE", n=int(IDX["_seq"].isna().sum()))
bad = set(IDX["_split"]) - set(EXPECTED)
if bad: stop("M2ENC_UNKNOWN_SPLIT", values=sorted(bad))
IDX["_record_key"] = IDX["_split"] + "/" + IDX["_bearing"]

# 레코드 → npy 파일 연결
def find_npy(split, bearing, idx_file):
    sib = sorted(Path(idx_file).parent.glob("*.npy")) if len(INDEX_FILES) > 1 else []
    if len(sib) == 1: return sib[0]
    hits = []
    for p in NPY_FILES:
        s = str(p.relative_to(DATASET_DIR)).upper()
        if not re.search(rf"BEARING{bearing[-3]}_{bearing[-1]}(?!\d)", s): continue
        is_full = ("FULL_TEST" in s) or ("FULLTEST" in s)
        if split == "FULL_TEST" and is_full: hits.append(p)
        elif split == "TEST" and ("TEST" in s) and not is_full: hits.append(p)
        elif split == "LEARNING" and "LEARNING" in s: hits.append(p)
    if len(hits) != 1:
        stop("M2ENC_NPY_MAPPING_AMBIGUOUS", record=f"{split}/{bearing}", hits=[str(h) for h in hits])
    return hits[0]

rec_rows, WIN = [], {}
for key, g in IDX.groupby("_record_key", sort=True):
    g = g.sort_values("_seq")
    split, bearing = g["_split"].iloc[0], g["_bearing"].iloc[0]
    if c_npy:
        vals = g[c_npy].astype(str).str.strip().unique()
        if len(vals) != 1: stop("M2ENC_NPY_COLUMN_NOT_UNIQUE", record=key, n=len(vals))
        npy = Path(vals[0]); npy = npy if npy.is_absolute() else DATASET_DIR / npy
    else:
        npy = find_npy(split, bearing, g["_index_file"].iloc[0])
    shape = np.load(npy, mmap_mode="r").shape
    rows = g[c_row].astype(int).to_numpy() if c_row else np.arange(len(g))
    if len(shape) != 2 or shape[1] != NS or rows.max() >= shape[0] or (not c_row and shape[0] != len(g)):
        stop("M2ENC_NPY_SHAPE_MISMATCH", record=key, npy_shape=shape, n_index=len(g))
    role = {"FULL_TEST": "TRAIN", "LEARNING": "EVAL", "TEST": "EXCLUDED"}[split]
    rec_rows.append(dict(record_key=key, split=split, bearing=bearing, n=len(g), role=role,
                         fold=FOLDS.get(bearing) if role == "TRAIN" else None,
                         npy_path=str(npy), npy_rows=shape[0],
                         note="FULL_TEST 앞부분 중복/#9 보류(D-026)" if role == "EXCLUDED" else ""))
    WIN[key] = rows
RECORDS = pd.DataFrame(rec_rows)

# 범위 검사
chk = {}
for sp, (n_exp, r_exp) in EXPECTED.items():
    sub = RECORDS[RECORDS.split == sp]
    chk[sp] = (int(sub.n.sum()), len(sub))
    if chk[sp] != (n_exp, r_exp): stop("M2ENC_SCOPE_MISMATCH", got=chk, expected=EXPECTED)
tr_b = set(RECORDS[RECORDS.role == "TRAIN"].bearing)
ev_b = set(RECORDS[RECORDS.role == "EVAL"].bearing)
if tr_b & ev_b: stop("M2ENC_TRAIN_EVAL_OVERLAP", overlap=sorted(tr_b & ev_b))
if tr_b != set(FOLDS): stop("M2ENC_FOLD_TABLE_MISMATCH", train=sorted(tr_b), folds=sorted(FOLDS))

RUN_DIR = OUT_ROOT / (datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_m2_enc_v1_data_prep")
RUN_DIR.mkdir(parents=True, exist_ok=False)
mp = RUN_DIR / "split_manifest.csv"
RECORDS.to_csv(mp, index=False, encoding="utf-8-sig")
SPLIT_SHA = hashlib.sha256(mp.read_bytes()).hexdigest()

print("[SCOPE]", chk)
print(RECORDS[["record_key","role","fold","n"]].to_string(index=False))
print(f"\n[SPLIT] TRAIN {int(RECORDS[RECORDS.role=='TRAIN'].n.sum())}윈도 / "
      f"EVAL {int(RECORDS[RECORDS.role=='EVAL'].n.sum())} / EXCLUDED {int(RECORDS[RECORDS.role=='EXCLUDED'].n.sum())}")
print("[SPLIT] 학습∩평가 베어링 = 없음 ✅")
print("[OUT]", RUN_DIR, "\n[SHA256] split_manifest.csv", SPLIT_SHA)


[SCOPE] {'LEARNING': (7534, 6), 'TEST': (14127, 11), 'FULL_TEST': (17355, 11)}
          record_key     role  fold    n
FULL_TEST/Bearing1_3    TRAIN   0.0 2375
FULL_TEST/Bearing1_4    TRAIN   1.0 1428
FULL_TEST/Bearing1_5    TRAIN   2.0 2463
FULL_TEST/Bearing1_6    TRAIN   3.0 2448
FULL_TEST/Bearing1_7    TRAIN   4.0 2259
FULL_TEST/Bearing2_3    TRAIN   0.0 1955
FULL_TEST/Bearing2_4    TRAIN   1.0  751
FULL_TEST/Bearing2_5    TRAIN   2.0 2311
FULL_TEST/Bearing2_6    TRAIN   3.0  701
FULL_TEST/Bearing2_7    TRAIN   4.0  230
FULL_TEST/Bearing3_3    TRAIN   2.0  434
 LEARNING/Bearing1_1     EVAL   NaN 2803
 LEARNING/Bearing1_2     EVAL   NaN  871
 LEARNING/Bearing2_1     EVAL   NaN  911
 LEARNING/Bearing2_2     EVAL   NaN  797
 LEARNING/Bearing3_1     EVAL   NaN  515
 LEARNING/Bearing3_2     EVAL   NaN 1637
     TEST/Bearing1_3 EXCLUDED   NaN  590
     TEST/Bearing1_4 EXCLUDED   NaN 1139
     TEST/Bearing1_5 EXCLUDED   NaN 2302
     TEST/Bearing1_6 EXCLUDED   NaN 2232
     TEST/Bearing1_

In [ ]:
# ===== M2_ENC_v1 Cell 3: LP6000 + 스케일 s(D-027) + INT8 범위 Q 후보 측정 (학습 레코드만) =====
from scipy import signal

# 필터 재설계 + Next-4 수치와 대조 (게이트)
numtaps, beta = signal.kaiserord(60.0, 400.0 / (FS / 2))
if numtaps % 2 == 0: numtaps += 1
H_LP = signal.firwin(numtaps, 6000.0, window=("kaiser", beta), fs=FS)
_, Hf = signal.freqz(H_LP, worN=[5800.0, 6000.0, 6200.0], fs=FS)
resp = 20 * np.log10(np.abs(Hf))
filt_ok = (numtaps == 187 and abs(beta - 5.653) < 1e-3 and abs(resp[0] + 0.01) < 0.05
           and abs(resp[1] + 6.02) < 0.05 and resp[2] <= -60.0)
key_try = hashlib.sha256(H_LP.astype(np.float64).tobytes()).hexdigest()[:12]
print(f"[FILT] taps={numtaps} beta={beta:.3f} 응답 5800/6000/6200Hz = {resp.round(2).tolist()} dB")
print(f"[FILT] 계수 해시={key_try} (Next-4 FILT_KEY 7e32cc3e140c와 산출식이 다를 수 있음, 참고용)")
if not filt_ok: stop("M2ENC_FILTER_MISMATCH", taps=numtaps, beta=beta, resp=resp.tolist())
PAD = (numtaps - 1) // 2

def lp6000(X):
    Xp = np.pad(X.astype(np.float64), ((0, 0), (PAD, PAD)), mode="reflect")
    return signal.fftconvolve(Xp, H_LP[None, :], mode="valid", axes=1)

def load_record(r, tries=5):
    for t in range(tries):
        try:
            A = np.load(r.npy_path, mmap_mode="r")
            return np.asarray(A[WIN[r.record_key]], dtype=np.float32)
        except OSError as e:
            print(f"   재시도 {t+1}/{tries}: {r.record_key} {e}"); time.sleep(5 * (t + 1))
    stop("M2ENC_READ_FAILED", record=r.record_key)

rows, all_pool, norm_pool, tail_pool = [], [], [], []
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all():
        stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X)
    rms = np.sqrt((Y ** 2).mean(axis=1))
    s = float(np.median(rms[:REG_MIN]))
    Z = np.abs(Y / s).astype(np.float32)
    nt = max(1, int(round(0.1 * r.n)))
    all_pool.append(Z.ravel()); norm_pool.append(Z[:REG_MIN].ravel()); tail_pool.append(Z[-nt:].ravel())
    rows.append(dict(record_key=r.record_key, fold=r.fold, n=r.n, s=s,
                     dc_ratio=float(np.abs(Y[:REG_MIN].mean()) / s),
                     tail_rms_ratio=float(np.median(rms[-nt:]) / s),
                     absz_p999_normal=float(np.percentile(Z[:REG_MIN], 99.9)),
                     absz_max=float(Z.max())))
    print(f"  {r.record_key:22s} n={r.n:5d} s={s:.5f} 말기RMS/s={rows[-1]['tail_rms_ratio']:.2f}")
    del X, Y, Z

SCALE = pd.DataFrame(rows)
A, N_, T_ = (np.concatenate(p) for p in (all_pool, norm_pool, tail_pool))
qc = {"ALL_p99.9": np.percentile(A, 99.9), "ALL_p99.99": np.percentile(A, 99.99), "ALL_max": A.max(),
      "NORMAL_p99.9": np.percentile(N_, 99.9), "NORMAL_max": N_.max()}
n999 = np.percentile(N_, 99.9)
QTAB = pd.DataFrame([dict(rule=k, Q=float(q),
                          normal_levels_used=int(2 * min(127, round(127 * n999 / q)) + 1),
                          clip_rate_normal=float((N_ > q).mean()),
                          clip_rate_tail10=float((T_ > q).mean()),
                          clip_rate_all=float((A > q).mean())) for k, q in qc.items()])

SCALE.to_csv(RUN_DIR / "scale_by_record.csv", index=False, encoding="utf-8-sig")
QTAB.to_csv(RUN_DIR / "q_candidates.csv", index=False, encoding="utf-8-sig")
summary = dict(status="M2ENC_DATA_PREP_MEASURED_AWAITING_Q_RULE", contract=CONTRACT_VERSION,
               run_dir=str(RUN_DIR), split_manifest_sha256=SPLIT_SHA,
               filter=dict(taps=int(numtaps), beta=float(beta), resp_db=resp.round(2).tolist(),
                           coef_hash_try=key_try),
               train_windows=int(SCALE.n.sum()), learning_signals_read=False,
               baseline_d025=BASELINE_D025, criteria_d028=CRITERIA_D028)
(RUN_DIR / "DATA_PREP_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pd.set_option("display.width", 200)
print("\n[SCALE]\n", SCALE.round(4).to_string(index=False))
print("\n[Q 후보] (정상 사용 단계 수: 256단계 중 정상 구간이 실제로 쓰는 단계)\n", QTAB.round(5).to_string(index=False))
print("\nSTATUS:", summary["status"], "| LEARNING 신호 읽기: 없음")


[FILT] taps=187 beta=5.653 응답 5800/6000/6200Hz = [-0.01, -6.02, -60.96] dB
[FILT] 계수 해시=b5543b5a8f89 (Next-4 FILT_KEY 7e32cc3e140c와 산출식이 다를 수 있음, 참고용)
  FULL_TEST/Bearing1_3   n= 2375 s=0.36424 말기RMS/s=4.87
  FULL_TEST/Bearing1_4   n= 1428 s=0.39174 말기RMS/s=14.77
  FULL_TEST/Bearing1_5   n= 2463 s=0.32944 말기RMS/s=0.72
  FULL_TEST/Bearing1_6   n= 2448 s=0.41906 말기RMS/s=0.70
  FULL_TEST/Bearing1_7   n= 2259 s=0.42610 말기RMS/s=0.83
  FULL_TEST/Bearing2_3   n= 1955 s=0.47478 말기RMS/s=0.50
  FULL_TEST/Bearing2_4   n=  751 s=0.34546 말기RMS/s=0.77
  FULL_TEST/Bearing2_5   n= 2311 s=0.65251 말기RMS/s=0.36
  FULL_TEST/Bearing2_6   n=  701 s=0.28748 말기RMS/s=0.69
  FULL_TEST/Bearing2_7   n=  230 s=0.33397 말기RMS/s=1.14
  FULL_TEST/Bearing3_3   n=  434 s=0.25796 말기RMS/s=0.84

[SCALE]
           record_key  fold    n      s  dc_ratio  tail_rms_ratio  absz_p999_normal  absz_max
FULL_TEST/Bearing1_3   0.0 2375 0.3642    0.0018          4.8670            3.7022  149.4143
FULL_TEST/Bearing1_4   1.0 1428 0.39

In [ ]:
# ===== M2_ENC_v1 Cell 3B: FILT_KEY 산출식 역확인 (읽기 전용, 쓰기 없음) =====
TARGET = "7e32cc3e140c"
h = lambda b: hashlib.sha256(b).hexdigest()[:12]
P = dict(fs=FS, cutoff_hz=6000.0, atten_db=60.0, transition_hz=400.0,
         numtaps=int(numtaps), beta=float(beta), window="kaiser")
cand = {
  "coef_float64": h(H_LP.astype(np.float64).tobytes()),
  "coef_float32": h(H_LP.astype(np.float32).tobytes()),
  "coef_round12": h(np.round(H_LP, 12).tobytes()),
  "coef_repr":    h(repr(H_LP.tolist()).encode()),
  "coef_json":    h(json.dumps(H_LP.tolist()).encode()),
}
for name, keys in {"params_all": list(P), "params_core": ["fs","cutoff_hz","atten_db","transition_hz"]}.items():
    d = {k: P[k] for k in keys}
    cand[name+"_sorted"]  = h(json.dumps(d, sort_keys=True).encode())
    cand[name+"_compact"] = h(json.dumps(d, sort_keys=True, separators=(",", ":")).encode())
for k, v in cand.items():
    print(f"{'✅' if v == TARGET else '  '} {k:24s} {v}")
hit = [k for k, v in cand.items() if v == TARGET]
print("\n결과:", f"일치 → {hit}" if hit else "일치 없음 → Next-4 노트북에서 'FILT_KEY =' 계산 줄 1개 필요")


   coef_float64             b5543b5a8f89
   coef_float32             d7e6919db883
   coef_round12             bea94475c39c
   coef_repr                42b7c82a3926
   coef_json                42b7c82a3926
   params_all_sorted        29403e04d7b1
   params_all_compact       0a2f703c9ca5
   params_core_sorted       aa21153c644e
   params_core_compact      bac9662d06d3

결과: 일치 없음 → Next-4 노트북에서 'FILT_KEY =' 계산 줄 1개 필요


In [ ]:
# ===== M2_ENC_v1 Cell 4: output_row_index 전수 검증 + 학습 데이터 적재 (LP6000 → x/s → clip ±Q) =====
Q_IN = 8.0                       # D-029
EXPECTED_SPLIT_SHA = "41a0e1f243eb1fa98b4d8dfc894b281ab843e6faf7f9d5902b7f0a97d790c10e"
if SPLIT_SHA != EXPECTED_SPLIT_SHA:
    stop("M2ENC_SPLIT_CHANGED", got=SPLIT_SHA, expected=EXPECTED_SPLIT_SHA)

for c in ["output_array_relative_path", "output_row_index"]:
    if c not in IDX.columns: stop("M2ENC_COLUMN_NOT_FOUND", what=c, available=list(IDX.columns))

map_issues = []
for r in RECORDS.itertuples():
    g = IDX[IDX._record_key == r.record_key].sort_values("_seq")
    seq = g._seq.to_numpy()
    rows = g.output_row_index.astype(int).to_numpy()
    paths = g.output_array_relative_path.astype(str).str.strip().unique()
    p_ok = len(paths) == 1 and os.path.normpath(str(DATASET_DIR / paths[0])) == os.path.normpath(r.npy_path)
    if not p_ok or not np.all(np.diff(seq) > 0) or rows.min() < 0 or rows.max() >= r.npy_rows:
        stop("M2ENC_ROW_MAPPING_INVALID", record=r.record_key, paths=list(paths)[:3])
    if not np.array_equal(rows, WIN[r.record_key]):
        map_issues.append(r.record_key)
    WIN[r.record_key] = rows          # 이후는 output_row_index 기준
print(f"[MAP] 28개 레코드 검증 완료. Cell 2 가정(행순서=sequence)과 다른 레코드: {map_issues or '없음 ✅'}")
if map_issues:
    print("   ⚠ Cell 3의 s 값은 잘못된 행 순서로 계산됨 → 아래 s 재계산값을 기준으로 사용")

DATA, chk_rows = {}, []
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all():
        stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X)
    rms = np.sqrt((Y ** 2).mean(axis=1))
    s = float(np.median(rms[:REG_MIN]))
    Z = np.clip(Y / s, -Q_IN, Q_IN).astype(np.float32)
    DATA[r.record_key] = dict(Z=Z, t=np.arange(r.n) / (r.n - 1), s=s, fold=int(r.fold), n=r.n)
    s3 = float(SCALE.loc[SCALE.record_key == r.record_key, "s"].iloc[0])
    chk_rows.append(dict(record_key=r.record_key, fold=int(r.fold), n=r.n, s=s,
                         s_cell3=s3, s_match=abs(s - s3) <= 1e-9 * max(1.0, abs(s3)),
                         clip_rate=float((np.abs(Y / s) > Q_IN).mean())))
    del X, Y
CHK = pd.DataFrame(chk_rows)
print(CHK.round(6).to_string(index=False))
print(f"\n[DATA] 학습 윈도 {sum(v['n'] for v in DATA.values())}개, 메모리 약 "
      f"{sum(v['Z'].nbytes for v in DATA.values())/1e6:.0f}MB, 전체 clip 비율 "
      f"{np.average(CHK.clip_rate, weights=CHK.n):.5f}")
if not map_issues and not CHK.s_match.all():
    stop("M2ENC_SCALE_NOT_REPRODUCED", bad=CHK[~CHK.s_match].record_key.tolist())
print("Cell 4 완료")


[MAP] 28개 레코드 검증 완료. Cell 2 가정(행순서=sequence)과 다른 레코드: 없음 ✅
          record_key  fold    n        s  s_cell3  s_match  clip_rate
FULL_TEST/Bearing1_3     0 2375 0.364240 0.364240     True   0.017466
FULL_TEST/Bearing1_4     1 1428 0.391743 0.391743     True   0.090027
FULL_TEST/Bearing1_5     2 2463 0.329442 0.329442     True   0.000244
FULL_TEST/Bearing1_6     3 2448 0.419063 0.419063     True   0.000152
FULL_TEST/Bearing1_7     4 2259 0.426104 0.426104     True   0.000861
FULL_TEST/Bearing2_3     0 1955 0.474782 0.474782     True   0.001919
FULL_TEST/Bearing2_4     1  751 0.345464 0.345464     True   0.000373
FULL_TEST/Bearing2_5     2 2311 0.652514 0.652514     True   0.000004
FULL_TEST/Bearing2_6     3  701 0.287482 0.287482     True   0.000782
FULL_TEST/Bearing2_7     4  230 0.333969 0.333969     True   0.012511
FULL_TEST/Bearing3_3     2  434 0.257961 0.257961     True   0.001115

[DATA] 학습 윈도 17355개, 메모리 약 142MB, 전체 clip 비율 0.01042
Cell 4 완료


In [ ]:
# ===== M2_ENC_v1 Cell 5 (r2): 동결 M0 판정 함수 추출(재구현 금지) + 임베딩용 어댑터 =====
# [r2] 동결 노트북의 실행용 함수(run_m0 등)가 미정의 기본값(FEMTO_ROOT)을 참조해 NameError →
#      노드 단위 실행 실패는 기록 후 계속(COMPAT Cell 2와 동일 방식).
#      대신 register_and_detect가 참조하는 이름이 하나라도 미정의면 정지(안전장치).
#      판정 로직·상수 변경 없음.
import ast, inspect, json, hashlib, dis, builtins
import numpy as np
import pandas as pd

FROZEN_NB = PROJECT_ROOT / "M0_FEMTO_Baseline_v1.ipynb"
FROZEN_SHA_PREFIX = "921f1bbea7b460d748c6dc3f74c04981f3bf4de5ff76223b"
FROZEN_DETECT_FN = "register_and_detect"
EXPECTED_CONSTS = {"K": 6.0, "CONSEC": 5, "REG_MIN": 90, "REG_MAX": 600}

raw = FROZEN_NB.read_bytes()
nb_sha = hashlib.sha256(raw).hexdigest()
if not nb_sha.startswith(FROZEN_SHA_PREFIX):
    stop("M2ENC_FROZEN_NOTEBOOK_CHANGED", sha=nb_sha)

ns, skipped, bad_cells, exec_fail = {"__name__": "frozen_m0"}, [], 0, []
for ci, cell in enumerate(json.loads(raw.decode("utf-8"))["cells"]):
    if cell.get("cell_type") != "code": continue
    src = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("!", "%")))
    try: tree = ast.parse(src)
    except SyntaxError: bad_cells += 1; continue
    for node in tree.body:
        keep = isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef))
        if isinstance(node, ast.Assign) and all(isinstance(t, ast.Name) and t.id.isupper() for t in node.targets):
            try: ast.literal_eval(node.value); keep = True
            except Exception: skipped.append((ast.get_source_segment(src, node) or "")[:80])
        if keep:
            try:
                exec(compile(ast.Module([node], []), f"<frozen cell {ci}>", "exec"), ns)
            except Exception as e:   # [r2] 기록 후 계속
                name = getattr(node, "name", None) or (ast.get_source_segment(src, node) or "")[:60]
                exec_fail.append({"cell": ci, "name": name, "error": repr(e)[:120]})

if FROZEN_DETECT_FN not in ns:
    stop("M2ENC_FROZEN_FN_NOT_FOUND", available=sorted(k for k in ns if callable(ns[k]))[:50])
bad = {k: ns.get(k) for k, v in EXPECTED_CONSTS.items() if ns.get(k) != v}
if bad: stop("M2ENC_FROZEN_CONST_MISMATCH", got=bad, expected=EXPECTED_CONSTS)
FN_DET = ns[FROZEN_DETECT_FN]

# [r2] 안전장치: 판정 함수(및 그 안에서 부르는 동결 함수)가 참조하는 이름이 모두 정의돼 있는지 확인
def _names(code):
    out = {i.argval for i in dis.get_instructions(code) if i.opname in ("LOAD_GLOBAL", "LOAD_NAME")}
    for c in code.co_consts:
        if hasattr(c, "co_code"): out |= _names(c)
    return {str(n) for n in out}

def _unresolved(fn, seen=None):
    seen = set() if seen is None else seen
    if fn.__name__ in seen: return set()
    seen.add(fn.__name__); miss = set()
    for n in _names(fn.__code__):
        if n in ns:
            o = ns[n]
            if inspect.isfunction(o) and o.__globals__ is ns: miss |= _unresolved(o, seen)
        elif not hasattr(builtins, n):
            miss.add(n)
    return miss

_miss = sorted(_unresolved(FN_DET))
if _miss:
    stop("M2ENC_FROZEN_FN_UNRESOLVED", missing=_miss, exec_fail=exec_fail)

print(f"[FROZEN] sha={nb_sha[:16]}… , 파싱 불가 셀 {bad_cells}개, 실행 제외 대입문 {len(skipped)}개, 실행 실패 {len(exec_fail)}건")
if exec_fail:
    print("[FROZEN] 실행 실패 목록(판정 함수와 무관하면 정상 — 예: run_m0):")
    display(pd.DataFrame(exec_fail))
print(f"[FROZEN] {FROZEN_DETECT_FN}{inspect.signature(FN_DET)}  상수 {EXPECTED_CONSTS} ✅  미해결 참조 0건")

ONSET_KEYS  = ["onset", "onset_idx", "alarm_idx", "alarm_index", "first_alarm_idx"]
FAR_KEYS    = ["reg_far", "reg_FAR", "reg_far_ratio", "far_reg"]
STATUS_KEYS = ["status", "detected", "is_detected", "result"]

def _get(d, keys):
    for k in keys:
        if k in d: return k, d[k]
    return None, None

def _status_detected(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    s = str(v); sl = s.lower()
    if "미탐지" in s or "not" in sl or "miss" in sl: return False
    return ("탐지" in s) or ("detect" in sl)

def run_detector(E):
    out = FN_DET(np.asarray(E, dtype=np.float64))
    if not isinstance(out, dict):
        stop("M2ENC_DETECTOR_OUTPUT_NOT_DICT", type=str(type(out)), repr=repr(out)[:800])
    ko, onset = _get(out, ONSET_KEYS); kf, far = _get(out, FAR_KEYS); ks, st = _get(out, STATUS_KEYS)
    if kf is None or (ko is None and ks is None):
        stop("M2ENC_DETECTOR_KEYS_UNKNOWN", keys=list(out),
             sample={k: (v if np.isscalar(v) else type(v).__name__) for k, v in out.items()})
    det = onset is not None and not (isinstance(onset, float) and np.isnan(onset)) and int(onset) >= 0
    if ks is not None and _status_detected(st) != det:
        stop("M2ENC_DETECTOR_STATUS_MISMATCH", status=str(st), onset=str(onset))
    return dict(detected=bool(det), onset=int(onset) if det else None, reg_far=float(far))

# 스모크 테스트 (합성 데이터, 판정 기준 확인용 아님)
rng = np.random.default_rng(0)
E_syn = rng.normal(size=(600, 8)); E_syn[450:] += np.linspace(0, 8, 150)[:, None]
try:
    raw_out = FN_DET(E_syn)
except TypeError as e:
    stop("M2ENC_DETECTOR_SIGNATURE_UNKNOWN", sig=str(inspect.signature(FN_DET)), err=str(e))
print("[SMOKE] 원출력 키:", list(raw_out) if isinstance(raw_out, dict) else type(raw_out))
res = run_detector(E_syn)
print("[SMOKE] 어댑터 결과:", res, "" if res["detected"] else "※ 합성 변화 미검출(보고만)")
print("Cell 5 완료 (r2)")



[FROZEN] sha=921f1bbea7b460d7… , 파싱 불가 셀 0개, 실행 제외 대입문 3개, 실행 실패 1건
[FROZEN] 실행 실패 목록(판정 함수와 무관하면 정상 — 예: run_m0):


,cell,name,error
0,3,run_m0,"NameError(""name 'FEMTO_ROOT' is not defined"")"


[FROZEN] register_and_detect(feats, k=6.0, consec=5, reg_min=90, reg_max=600)  상수 {'K': 6.0, 'CONSEC': 5, 'REG_MIN': 90, 'REG_MAX': 600} ✅  미해결 참조 0건
[SMOKE] 원출력 키: ['n_reg', 'thr', 'dists', 'alarm_idx', 'N', 'reg_far']
[SMOKE] 어댑터 결과: {'detected': True, 'onset': 499, 'reg_far': 0.0} 
Cell 5 완료 (r2)


In [ ]:
# ===== M2_ENC_v1 Cell 6: 4구성 × 5-fold 학습 (D-030), 체크포인트로 이어하기 =====
import math, torch, torch.nn as nn
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("[DEVICE]", DEV, "" if DEV == "cuda" else "⚠ CPU: 매우 느림, GPU 런타임 권장")

HP = dict(epochs=30, batch=256, lr=1e-3, wd=1e-4, lam_compact=0.1,
          normal_frac=0.3, knee=0.7, seed=20261030, aug="roll+signflip", arch="v1a")
CONFIGS = [("LIN", 8), ("LIN", 16), ("KNEE", 8), ("KNEE", 16)]
TRAIN_KEY = hashlib.sha256(json.dumps(dict(split=SPLIT_SHA, Q=Q_IN, hp=HP, cfg=CONFIGS),
                                      sort_keys=True).encode()).hexdigest()[:12]
TRAIN_DIR = OUT_ROOT / f"m2_enc_v1_train_{TRAIN_KEY}"
(TRAIN_DIR / "ckpt").mkdir(parents=True, exist_ok=True)

def blk(ci, co, k, s, p):
    return nn.Sequential(nn.Conv1d(ci, co, k, s, p, bias=False), nn.BatchNorm1d(co), nn.ReLU())
class Enc(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.f = nn.Sequential(blk(1, 4, 12, 4, 4), blk(4, 8, 12, 4, 4), blk(8, 16, 6, 2, 2),
                               blk(16, 32, 6, 2, 2), blk(32, 32, 3, 1, 1),
                               nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.emb = nn.Linear(32, D)
    def forward(self, x): return self.emb(self.f(x))
class Net(nn.Module):
    def __init__(self, D):
        super().__init__(); self.enc = Enc(D); self.head = nn.Linear(D, 1)

def target(t, mode):
    return t if mode == "LIN" else np.clip((t - HP["knee"]) / (1 - HP["knee"]), 0, 1)

def fold_tensors(fold_out, mode):
    keys = [k for k, v in DATA.items() if v["fold"] != fold_out]
    cat = lambda f: np.concatenate([f(i, k) for i, k in enumerate(keys)])
    Z = cat(lambda i, k: DATA[k]["Z"])
    y = cat(lambda i, k: target(DATA[k]["t"], mode))
    rid = cat(lambda i, k: np.full(DATA[k]["n"], i))
    w = cat(lambda i, k: np.full(DATA[k]["n"], 1.0 / DATA[k]["n"])); w = w / w.mean()
    nm = cat(lambda i, k: DATA[k]["t"] < HP["normal_frac"])
    T = lambda a, dt: torch.as_tensor(a, dtype=dt, device=DEV)
    return T(Z, torch.float32), T(y, torch.float32), T(rid, torch.long), T(w, torch.float32), T(nm, torch.bool)

def compact_loss(e, r, m):
    e2, r2 = e[m], r[m]; parts = []
    for u in torch.unique(r2):
        eu = e2[r2 == u]
        if len(eu) >= 2: parts.append(((eu - eu.mean(0)) ** 2).sum(1).mean())
    if not parts: return e.new_zeros(())
    return torch.stack(parts).mean() / (e.var(0).sum() + 1e-6)   # 스케일 불변(붕괴 방지)

def train_one(mode, D, fold_out, seed):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    Zt, yt, rt, wt, nt = fold_tensors(fold_out, mode); N = len(yt)
    net = Net(D).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=HP["lr"], weight_decay=HP["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"] * math.ceil(N / HP["batch"]))
    hist = []
    for ep in range(HP["epochs"]):
        net.train(); perm = torch.randperm(N, generator=g).to(DEV); acc = [0.0, 0.0, 0]
        for i in range(0, N, HP["batch"]):
            idx = perm[i:i + HP["batch"]]
            x = torch.roll(Zt[idx].unsqueeze(1), int(torch.randint(0, NS, (1,), generator=g)), dims=-1)
            x = x * (torch.randint(0, 2, (len(idx), 1, 1), generator=g).float() * 2 - 1).to(DEV)
            e = net.enc(x); p = net.head(e).squeeze(-1)
            l_r = (wt[idx] * (p - yt[idx]) ** 2).mean(); l_c = compact_loss(e, rt[idx], nt[idx])
            loss = l_r + HP["lam_compact"] * l_c
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            acc[0] += l_r.item(); acc[1] += float(l_c); acc[2] += 1
        hist.append((ep, acc[0] / acc[2], acc[1] / acc[2]))
    return net, hist

n_params = sum(p.numel() for p in Enc(16).parameters())
print(f"[MODEL] 인코더 파라미터 {n_params}개 (D=16 기준), TRAIN_DIR={TRAIN_DIR}")
(TRAIN_DIR / "train_config.json").write_text(json.dumps(dict(train_key=TRAIN_KEY, hp=HP, configs=CONFIGS,
    Q_in=Q_IN, split_sha=SPLIT_SHA, folds=FOLDS, n_params_D16=n_params), ensure_ascii=False, indent=2), encoding="utf-8")

t0 = time.time()
for ci, (mode, D) in enumerate(CONFIGS):
    for f in range(5):
        ck = TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{f}.pt"
        if ck.exists(): print(f"  RESUME SKIP {ck.name}"); continue
        ts = time.time()
        net, hist = train_one(mode, D, f, HP["seed"] + 100 * ci + f)
        tmp = ck.with_suffix(".tmp")
        torch.save(dict(state=net.state_dict(), mode=mode, D=D, fold=f, hist=hist, hp=HP), tmp)
        os.replace(tmp, ck)
        print(f"  {ck.name}: 회귀손실 {hist[-1][1]:.4f}, 응집손실 {hist[-1][2]:.4f}, {time.time()-ts:.0f}s")
print(f"Cell 6 완료 ({(time.time()-t0)/60:.1f}분)")


[DEVICE] cuda 
[MODEL] 인코더 파라미터 8056개 (D=16 기준), TRAIN_DIR=/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/m2_enc_v1_train_42f861b9506e


/tmp/ipykernel_800/2341842389.py:67: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  acc[0] += l_r.item(); acc[1] += float(l_c); acc[2] += 1


  LIN_D8_f0.pt: 회귀손실 0.0178, 응집손실 0.0049, 21s
  LIN_D8_f1.pt: 회귀손실 0.0193, 응집손실 0.0072, 16s
  LIN_D8_f2.pt: 회귀손실 0.0146, 응집손실 0.0034, 12s
  LIN_D8_f3.pt: 회귀손실 0.0192, 응집손실 0.0061, 15s
  LIN_D8_f4.pt: 회귀손실 0.0176, 응집손실 0.0047, 16s
  LIN_D16_f0.pt: 회귀손실 0.0179, 응집손실 0.0024, 14s
  LIN_D16_f1.pt: 회귀손실 0.0204, 응집손실 0.0031, 16s
  LIN_D16_f2.pt: 회귀손실 0.0139, 응집손실 0.0042, 12s
  LIN_D16_f3.pt: 회귀손실 0.0189, 응집손실 0.0034, 15s
  LIN_D16_f4.pt: 회귀손실 0.0174, 응집손실 0.0034, 16s
  KNEE_D8_f0.pt: 회귀손실 0.0337, 응집손실 0.0028, 13s
  KNEE_D8_f1.pt: 회귀손실 0.0361, 응집손실 0.0027, 16s
  KNEE_D8_f2.pt: 회귀손실 0.0211, 응집손실 0.0014, 12s
  KNEE_D8_f3.pt: 회귀손실 0.0334, 응집손실 0.0015, 15s
  KNEE_D8_f4.pt: 회귀손실 0.0374, 응집손실 0.0026, 16s
  KNEE_D16_f0.pt: 회귀손실 0.0311, 응집손실 0.0033, 13s
  KNEE_D16_f1.pt: 회귀손실 0.0368, 응집손실 0.0048, 16s
  KNEE_D16_f2.pt: 회귀손실 0.0223, 응집손실 0.0019, 12s
  KNEE_D16_f3.pt: 회귀손실 0.0324, 응집손실 0.0015, 15s
  KNEE_D16_f4.pt: 회귀손실 0.0386, 응집손실 0.0019, 16s
Cell 6 완료 (4.9분)


In [ ]:
# ===== M2_ENC_v1 Cell 7: fold 검증(동결 판정) + D-030 선택 규칙. LEARNING 미사용 =====
@torch.no_grad()
def embed(net, Z, bs=512):
    net.eval(); out = []
    for i in range(0, len(Z), bs):
        out.append(net.enc(torch.from_numpy(Z[i:i + bs]).unsqueeze(1).to(DEV)).double().cpu().numpy())
    return np.concatenate(out)

rows = []
for mode, D in CONFIGS:
    for f in range(5):
        ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{f}.pt", map_location=DEV)
        net = Net(D).to(DEV); net.load_state_dict(ck["state"])
        for k, v in DATA.items():
            if v["fold"] != f: continue
            E = embed(net, v["Z"]); res = run_detector(E)
            lead = (v["n"] - res["onset"]) * 10 / 3600 if res["detected"] else np.nan
            rows.append(dict(config=f"{mode}_D{D}", mode=mode, D=D, fold=f, record_key=k, n=v["n"],
                             detected=res["detected"], onset=res["onset"], lead_h=lead,
                             reg_far_pct=100 * res["reg_far"], emb_min_std=float(E[:REG_MIN].std(0).min())))
VAL = pd.DataFrame(rows)
VAL.to_csv(TRAIN_DIR / "fold_validation.csv", index=False, encoding="utf-8-sig")

AGG = VAL.groupby(["config", "mode", "D"]).agg(detected=("detected", "sum"), n_rec=("record_key", "count"),
        mean_lead_h=("lead_h", "mean"), mean_reg_far_pct=("reg_far_pct", "mean"),
        min_emb_std=("emb_min_std", "min")).reset_index()
best_det = AGG.detected.max()
cand = AGG[(AGG.detected == best_det) & (AGG.mean_reg_far_pct <= CRITERIA_D028["mean_reg_far_pct_max"])]
if cand.empty:
    status, sel = "M2ENC_SELECTION_NO_CANDIDATE", None
else:
    sel = cand.sort_values(["mean_lead_h", "D"], ascending=[False, True]).iloc[0]
    status = "M2ENC_CV_SELECTED_AWAITING_FINAL"

summary = dict(status=status, train_key=TRAIN_KEY, train_dir=str(TRAIN_DIR),
               selected=None if sel is None else dict(config=sel.config, mode=sel["mode"], D=int(sel.D)),
               rule="max detected → mean_reg_far≤0.924% → max mean_lead → smaller D",
               learning_evaluated=False, unresolved_12_open=True,
               aggregate=AGG.round(4).to_dict(orient="records"))
(TRAIN_DIR / "CV_SELECTION_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 200)
print(VAL.pivot_table(index="record_key", columns="config", values="onset", aggfunc="first").to_string())
print("\n[AGG] (11개 학습 레코드, 각 레코드는 자기 fold 모델로 1회 검증)\n", AGG.round(4).to_string(index=False))
print("\nSTATUS:", status, "| 선택:", None if sel is None else sel.config, "| LEARNING 평가: 안 함")


config                KNEE_D16  KNEE_D8  LIN_D16  LIN_D8
record_key                                              
FULL_TEST/Bearing1_3    1117.0   1254.0   1319.0  1219.0
FULL_TEST/Bearing1_4     756.0    880.0    653.0   692.0
FULL_TEST/Bearing1_5    1276.0   1575.0    920.0  1101.0
FULL_TEST/Bearing1_6     841.0   1591.0   1234.0  1587.0
FULL_TEST/Bearing1_7    1741.0    959.0   1169.0  2211.0
FULL_TEST/Bearing2_3    1944.0   1944.0   1929.0  1937.0
FULL_TEST/Bearing2_4     397.0    485.0    397.0   492.0
FULL_TEST/Bearing2_6     350.0    411.0    355.0   428.0
FULL_TEST/Bearing2_7     206.0    223.0    176.0   182.0
FULL_TEST/Bearing3_3     423.0    423.0    249.0   324.0

[AGG] (11개 학습 레코드, 각 레코드는 자기 fold 모델로 1회 검증)
   config mode  D  detected  n_rec  mean_lead_h  mean_reg_far_pct  min_emb_std
KNEE_D16 KNEE 16        10     11       1.6647            0.0933       0.0072
 KNEE_D8 KNEE  8        10     11       1.4719            0.1711       0.0135
 LIN_D16  LIN 16        10     11  

In [ ]:
# ===== M2_ENC_v1 Cell 7B (r2): 진단 전용 — 같은 FULL_TEST 11개에서 M0-PHY(LP6000) vs M2-ENC(선택 구성) =====
# [r2] 동결 extract_features(sig, fs=25600) → 실제 입력 fs(20480) 명시 전달 + fs 민감도 자동 확인.
# D-030 선택 규칙·결과 변경 없음. LEARNING 미사용. 쓰기: TRAIN_DIR 안 진단 파일 2개만.
FROZEN_FEATURE_FN = "extract_features"
if sel is None: stop("M2ENC_DIAG_NO_SELECTION")
if FROZEN_FEATURE_FN not in ns:
    stop("M2ENC_FROZEN_FEATURE_FN_NOT_FOUND", available=sorted(k for k in ns if callable(ns[k]))[:50])
FN_FEAT = ns[FROZEN_FEATURE_FN]
sig = inspect.signature(FN_FEAT)
print("[M0] 동결 특징 함수:", FROZEN_FEATURE_FN, sig)

FS_PARAM = next((p for p in sig.parameters if p.lower() == "fs"), None)
FEAT_KW = {FS_PARAM: FS} if FS_PARAM else {}          # FS = 20480 (Cell 1)

def _feat_raw(x, **kw):
    out = FN_FEAT(np.asarray(x, dtype=np.float64), **kw)
    return np.asarray(list(out.values()) if isinstance(out, dict) else out, dtype=np.float64).ravel()

def feat9(x): return _feat_raw(x, **FEAT_KW)

# fs 민감도 확인: 같은 신호를 fs=20480 / fs=25600으로 계산해 비교
_x = lp6000(np.random.default_rng(1).normal(size=(1, NS)))[0]
try:
    _v = feat9(_x)
    _v_25k = _feat_raw(_x, **{FS_PARAM: 25600}) if FS_PARAM else _v
except Exception as e:
    stop("M2ENC_FEATURE_CALL_FAILED", sig=str(sig), err=repr(e)[:300])
if _v.size != 9: stop("M2ENC_FEATURE_DIM_UNEXPECTED", dim=int(_v.size), sig=str(sig))
FS_SENSITIVE = not np.allclose(_v, _v_25k, rtol=1e-12, atol=0.0)
_diff_idx = [int(i) for i in np.where(~np.isclose(_v, _v_25k, rtol=1e-12, atol=0.0))[0]]
print(f"[M0] fs 전달: {FEAT_KW or '없음'} | fs 민감도: "
      + (f"있음 ⚠ (특징 인덱스 {_diff_idx}가 fs에 따라 달라짐 → D-025 기준선 fs 확인 필요 [확인 대기])"
         if FS_SENSITIVE else "없음 ✅ (9특징 모두 fs 무관 — 결과 그대로 유효)"))

def diag(F_or_E, n):
    raw = FN_DET(np.asarray(F_or_E, dtype=np.float64)); res = run_detector(F_or_E)
    d = np.asarray(raw.get("dists", []), dtype=np.float64); thr = float(raw.get("thr", np.nan))
    on = res["onset"]
    post = float((d[on:] > thr).mean()) if (res["detected"] and len(d) == n) else np.nan
    return dict(detected=res["detected"], onset=on,
                onset_frac=(on / n) if res["detected"] else np.nan,
                lead_h=((n - on) * 10 / 3600) if res["detected"] else np.nan,
                reg_far_pct=100 * res["reg_far"], post_exceed=post)

mode_s, D_s = sel["mode"], int(sel.D)
rows7b, t0 = [], time.time()
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r); Y = lp6000(X)                      # M0: 2048 + LP6000 (클리핑 없음)
    F = np.stack([feat9(y) for y in Y]); del X, Y
    ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode_s}_D{D_s}_f{int(r.fold)}.pt", map_location=DEV)
    net = Net(D_s).to(DEV); net.load_state_dict(ck["state"])
    E = embed(net, DATA[r.record_key]["Z"])
    for tag, M in (("M0_PHY", F), (f"M2_{mode_s}_D{D_s}", E)):
        rows7b.append(dict(record_key=r.record_key, model=tag, n=r.n,
                           tail_rms_ratio=float(SCALE.loc[SCALE.record_key == r.record_key, "tail_rms_ratio"].iloc[0]),
                           **diag(M, r.n)))
    print(f"  {r.record_key:22s} 완료 ({time.time()-t0:.0f}s)")

DIAG = pd.DataFrame(rows7b)
DIAG.to_csv(TRAIN_DIR / "cv_diag_m0_vs_m2.csv", index=False, encoding="utf-8-sig")
AGG7 = DIAG.groupby("model").agg(detected=("detected", "sum"), mean_lead_h=("lead_h", "mean"),
                                 mean_reg_far_pct=("reg_far_pct", "mean"),
                                 median_onset_frac=("onset_frac", "median"),
                                 mean_post_exceed=("post_exceed", "mean")).reset_index()
(TRAIN_DIR / "CV_DIAG_SUMMARY.json").write_text(json.dumps(dict(
    status="M2ENC_CV_DIAG_COMPLETE", selection_unchanged=True, selected=sel.config,
    feature_fn_sig=str(sig), feature_fs_passed=FEAT_KW, feature_fs_sensitive=bool(FS_SENSITIVE),
    feature_fs_sensitive_idx=_diff_idx,
    m0_baseline_fs_match="CONFIRM_PENDING" if FS_SENSITIVE else "NOT_APPLICABLE",
    learning_evaluated=False,
    aggregate=AGG7.round(4).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 220)
print("\n[레코드별]\n", DIAG.pivot_table(index=["record_key", "tail_rms_ratio"], columns="model",
      values=["onset_frac", "post_exceed"], aggfunc="first").round(3).to_string())
print("\n[요약]\n", AGG7.round(4).to_string(index=False))
print("\nSTATUS: M2ENC_CV_DIAG_COMPLETE | 선택 변경 없음 | LEARNING 평가: 안 함"
      + (" | ⚠ M0 fs 민감 → [확인 대기]" if FS_SENSITIVE else ""))



[M0] 동결 특징 함수: extract_features (sig, fs=25600)
[M0] fs 전달: {'fs': 20480} | fs 민감도: 있음 ⚠ (특징 인덱스 [8]가 fs에 따라 달라짐 → D-025 기준선 fs 확인 필요 [확인 대기])
  FULL_TEST/Bearing1_3   완료 (6s)
  FULL_TEST/Bearing1_4   완료 (9s)
  FULL_TEST/Bearing1_5   완료 (13s)
  FULL_TEST/Bearing1_6   완료 (18s)
  FULL_TEST/Bearing1_7   완료 (22s)
  FULL_TEST/Bearing2_3   완료 (25s)
  FULL_TEST/Bearing2_4   완료 (26s)
  FULL_TEST/Bearing2_5   완료 (30s)
  FULL_TEST/Bearing2_6   완료 (32s)
  FULL_TEST/Bearing2_7   완료 (32s)
  FULL_TEST/Bearing3_3   완료 (33s)

[레코드별]
                                     onset_frac            post_exceed           
model                                   M0_PHY M2_LIN_D16      M0_PHY M2_LIN_D16
record_key           tail_rms_ratio                                             
FULL_TEST/Bearing1_3 4.867040            0.560      0.555       0.987      0.973
FULL_TEST/Bearing1_4 14.774470           0.757      0.457       1.000      0.955
FULL_TEST/Bearing1_5 0.715158            0.978      0.374       1.000  

In [ ]:
# ===== M2_ENC_v1 Cell 7C: 진단 전용 — #12 FILT_KEY 확인 + 조기 경보가 '물리 변화'인지 '시간 드리프트'인지 =====
# D-030 선택·결과 변경 없음. LEARNING 미사용. 쓰기: TRAIN_DIR 안 진단 파일 2개만.
from scipy.stats import spearmanr

# --- (1) #12: Next-4 공식 그대로 FILT_KEY 재계산 ---
N4_CUT = {"B_LP_NONE": None, "B_LP8000": 8000.0, "B_LP6000": 6000.0}
def _n4_key(fs_val):
    return hashlib.sha256(json.dumps({"fs": fs_val, "cut": N4_CUT, "ripple": 60.0, "width": 400.0,
        "numtaps": int(numtaps), "beta": float(beta)}, sort_keys=True).encode()).hexdigest()[:12]
k_int, k_flt = _n4_key(int(FS)), _n4_key(float(FS))
FILT_KEY_OK = "7e32cc3e140c" in (k_int, k_flt)
print(f"[#12] Next-4 공식 재계산: fs=int → {k_int} / fs=float → {k_flt} | 7e32cc3e140c 일치: {'✅' if FILT_KEY_OK else '❌'}")
print(f"[#12] LP6000 계수 해시(실제 계수 식별용): {key_try}")

# --- (2) 드리프트 진단 ---
if sel is None: stop("M2ENC_DIAG_NO_SELECTION")
def _sp(a, b):
    r = spearmanr(a, b).correlation
    return float(r) if np.isfinite(r) else np.nan

def _metrics(M, n, t, d0, thr0):
    raw = FN_DET(np.asarray(M, dtype=np.float64)); res = run_detector(M)
    d = np.asarray(raw.get("dists", []), dtype=np.float64); thr = float(raw.get("thr", np.nan))
    nr = int(raw.get("n_reg", REG_MIN)); on = res["onset"]
    ok = len(d) == n and len(d0) == n
    return dict(detected=res["detected"],
                onset_frac=(on / n) if res["detected"] else np.nan,
                rho_time=_sp(d[nr:], t[nr:]) if ok else np.nan,          # 거리 ~ 시간
                rho_m0=_sp(d[nr:], d0[nr:]) if ok else np.nan,           # 거리 ~ M0 물리거리
                m0_ratio_at_onset=float(np.median(d0[on:on + 5]) / thr0)  # 경보 시점 M0 거리 / M0 임계
                    if (res["detected"] and ok) else np.nan)

MODELS = [("M2_" + sel.config, sel["mode"], int(sel.D))]
if sel.config != "KNEE_D16": MODELS.append(("M2_KNEE_D16", "KNEE", 16))
RAND_SEEDS = [0, 1, 2]                      # 학습 안 한 인코더 대조군

rows7c, t0 = [], time.time()
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    v = DATA[r.record_key]; n, t = v["n"], v["t"]
    X = load_record(r); Y = lp6000(X); F = np.stack([feat9(y) for y in Y]); del X, Y
    raw0 = FN_DET(F); d0 = np.asarray(raw0["dists"], dtype=np.float64); thr0 = float(raw0["thr"])
    base = dict(record_key=r.record_key, n=n,
                tail_rms_ratio=float(SCALE.loc[SCALE.record_key == r.record_key, "tail_rms_ratio"].iloc[0]))
    m0 = _metrics(F, n, t, d0, thr0); m0["rho_m0"] = 1.0
    rows7c.append({**base, "model": "M0_PHY", **m0})
    for tag, mode, D in MODELS:
        ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{int(r.fold)}.pt", map_location=DEV)
        net = Net(D).to(DEV); net.load_state_dict(ck["state"])
        rows7c.append({**base, "model": tag, **_metrics(embed(net, v["Z"]), n, t, d0, thr0)})
    for s in RAND_SEEDS:
        torch.manual_seed(1000 + s); net = Net(16).to(DEV)
        rows7c.append({**base, "model": f"RAND_D16_s{s}", **_metrics(embed(net, v["Z"]), n, t, d0, thr0)})
    print(f"  {r.record_key:22s} 완료 ({time.time()-t0:.0f}s)")

DR = pd.DataFrame(rows7c)
DR["group"] = DR.model.str.replace(r"_s\d$", "", regex=True)
DR.to_csv(TRAIN_DIR / "cv_diag_drift.csv", index=False, encoding="utf-8-sig")
AGG_DR = DR.groupby("group").agg(detected=("detected", "sum"), n=("record_key", "count"),
    median_onset_frac=("onset_frac", "median"), median_rho_time=("rho_time", "median"),
    median_rho_m0=("rho_m0", "median"), median_m0_ratio_at_onset=("m0_ratio_at_onset", "median")).reset_index()
(TRAIN_DIR / "CV_DRIFT_DIAG_SUMMARY.json").write_text(json.dumps(dict(
    status="M2ENC_CV_DRIFT_DIAG_COMPLETE", selection_unchanged=True, selected=sel.config,
    filt_key_recomputed={"fs_int": k_int, "fs_float": k_flt, "match_7e32cc3e140c": FILT_KEY_OK},
    lp6000_coef_hash=key_try, m0_feature_fs=FEAT_KW, learning_evaluated=False,
    aggregate=AGG_DR.round(4).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 240)
main = DR[~DR.model.str.startswith("RAND")]
print("\n[레코드별: 경보시점 / 경보 때 M0거리비]\n",
      main.pivot_table(index=["record_key", "tail_rms_ratio"], columns="model",
                       values=["onset_frac", "m0_ratio_at_onset"], aggfunc="first").round(3).to_string())
print("\n[레코드별: 거리~시간 상관 / 거리~M0 상관]\n",
      main.pivot_table(index="record_key", columns="model", values=["rho_time", "rho_m0"],
                       aggfunc="first").round(3).to_string())
print("\n[요약] (RAND는 3시드 합산)\n", AGG_DR.round(3).to_string(index=False))
print("\nSTATUS: M2ENC_CV_DRIFT_DIAG_COMPLETE | 선택 변경 없음 | LEARNING 평가: 안 함")


[#12] Next-4 공식 재계산: fs=int → 7e32cc3e140c / fs=float → 54b99597eff0 | 7e32cc3e140c 일치: ✅
[#12] LP6000 계수 해시(실제 계수 식별용): b5543b5a8f89
  FULL_TEST/Bearing1_3   완료 (8s)
  FULL_TEST/Bearing1_4   완료 (11s)
  FULL_TEST/Bearing1_5   완료 (16s)
  FULL_TEST/Bearing1_6   완료 (21s)
  FULL_TEST/Bearing1_7   완료 (25s)
  FULL_TEST/Bearing2_3   완료 (29s)
  FULL_TEST/Bearing2_4   완료 (30s)
  FULL_TEST/Bearing2_5   완료 (35s)
  FULL_TEST/Bearing2_6   완료 (36s)
  FULL_TEST/Bearing2_7   완료 (37s)
  FULL_TEST/Bearing3_3   완료 (38s)

[레코드별: 경보시점 / 경보 때 M0거리비]
                                     m0_ratio_at_onset                        onset_frac                       
model                                          M0_PHY M2_KNEE_D16 M2_LIN_D16     M0_PHY M2_KNEE_D16 M2_LIN_D16
record_key           tail_rms_ratio                                                                           
FULL_TEST/Bearing1_3 4.867040                   1.029       0.747      1.137      0.560       0.470      0.555
FULL_TEST/Bearing1_4 

In [ ]:
# ===== M2_ENC_v1 Cell 8: 최종 모델 재학습 (D-030: 선택 구성으로 FULL_TEST 11개 전체) =====
# LEARNING 미사용. 쓰기: TRAIN_DIR/ckpt/FINAL_*.pt 1개만.
if sel is None or sel.config != "LIN_D16":
    stop("M2ENC_FINAL_SELECTION_UNEXPECTED", sel=None if sel is None else sel.config)
FINAL_MODE, FINAL_D = sel["mode"], int(sel.D)
FINAL_SEED = HP["seed"] + 9999
FINAL_CK = TRAIN_DIR / "ckpt" / f"FINAL_{FINAL_MODE}_D{FINAL_D}.pt"

if FINAL_CK.exists():
    print("  RESUME SKIP", FINAL_CK.name)
else:
    ts = time.time()
    net, hist = train_one(FINAL_MODE, FINAL_D, -1, FINAL_SEED)   # fold_out=-1 → 11개 전부 학습
    tmp = FINAL_CK.with_suffix(".tmp")
    torch.save(dict(state=net.state_dict(), mode=FINAL_MODE, D=FINAL_D, fold="ALL", hist=hist,
                    hp=HP, seed=FINAL_SEED, train_records=sorted(DATA)), tmp)
    os.replace(tmp, FINAL_CK)
    print(f"  {FINAL_CK.name}: 회귀손실 {hist[-1][1]:.4f}, 응집손실 {hist[-1][2]:.4f}, {time.time()-ts:.0f}s")

FINAL_SHA = hashlib.sha256(FINAL_CK.read_bytes()).hexdigest()
_ck = torch.load(FINAL_CK, map_location=DEV)
if _ck["fold"] != "ALL" or len(_ck["train_records"]) != 11: stop("M2ENC_FINAL_CKPT_INVALID")
NET_F = Net(FINAL_D).to(DEV); NET_F.load_state_dict(_ck["state"]); NET_F.eval()

# 학습 데이터 자체 점검(성능 근거 아님 — 임베딩 붕괴만 확인)
_ins = []
for k, v in DATA.items():
    E = embed(NET_F, v["Z"]); _ins.append(dict(record_key=k, detected=run_detector(E)["detected"],
                                               emb_min_std=float(E[:REG_MIN].std(0).min())))
_ins = pd.DataFrame(_ins)
if (_ins.emb_min_std <= 0).any(): stop("M2ENC_FINAL_EMBEDDING_COLLAPSE", bad=_ins[_ins.emb_min_std <= 0].record_key.tolist())
print(f"[FINAL] {FINAL_CK.name}  sha256={FINAL_SHA[:16]}…  seed={FINAL_SEED}")
print(f"[FINAL] 학습데이터 자체 검출 {int(_ins.detected.sum())}/11 (참고용, 성능 근거 아님), 최소 emb_std {_ins.emb_min_std.min():.4f}")
print("Cell 8 완료")


  FINAL_LIN_D16.pt: 회귀손실 0.0179, 응집손실 0.0023, 20s
[FINAL] FINAL_LIN_D16.pt  sha256=508ad40a3fee3ee3…  seed=20271029
[FINAL] 학습데이터 자체 검출 10/11 (참고용, 성능 근거 아님), 최소 emb_std 0.0167
Cell 8 완료


In [ ]:
# ===== M2_ENC_v1 Cell 9: LEARNING 최종 평가 (D-030/D-031, 1회 한정) =====
# 순서: ① M0 재현 게이트(잠금 전) → ② 잠금 생성 → ③ M2 평가 → ④ D-028 판정 + 사전 고정 보고 항목
# 쓰기: TRAIN_DIR/learning_eval_once/ 안만. 동결 파일·원본·데이터셋 쓰기 없음.
from scipy.stats import spearmanr

_req = ["FN_DET", "run_detector", "ns", "lp6000", "load_record", "embed", "NET_F", "FINAL_SHA",
        "FINAL_CK", "RECORDS", "Q_IN", "nb_sha", "FROZEN_NB", "TRAIN_DIR"]
_miss = [n for n in _req if n not in globals()]
if _miss: stop("M2ENC_EVAL_PREREQ_MISSING", missing=_miss, note="Cell 1→2→3→4→5→6→7→8 순서로 실행")

# M0 특징: 동결 extract_features, fs=20480 명시 (Next-4와 동일 경로)
FN_FEAT = ns["extract_features"]
def feat9(x):
    out = FN_FEAT(np.asarray(x, dtype=np.float64), fs=20480)
    return np.asarray(list(out.values()) if isinstance(out, dict) else out, dtype=np.float64).ravel()
def _sp(a, b):
    r = spearmanr(a, b).correlation
    return float(r) if np.isfinite(r) else np.nan

D025_ONSET = {"Bearing1_1": 1462, "Bearing1_2": 826, "Bearing2_1": 874,
              "Bearing2_2": 487, "Bearing3_1": 496, "Bearing3_2": 1584}
D025_FAR   = {"Bearing1_1": 0.0033, "Bearing1_2": 0.0, "Bearing2_1": 0.0044,
              "Bearing2_2": 0.005, "Bearing3_1": 0.0078, "Bearing3_2": 0.0049}   # 소수 4자리 기록값

EVAL_DIR = TRAIN_DIR / "learning_eval_once"; EVAL_DIR.mkdir(exist_ok=True)
LOCK, SUMF = EVAL_DIR / "LEARNING_EVAL.lock", EVAL_DIR / "LEARNING_EVAL_SUMMARY.json"
if SUMF.exists():
    print(SUMF.read_text(encoding="utf-8"))
    stop("M2ENC_LEARNING_ALREADY_EVALUATED", note="1회 한정 — 위 저장 결과를 사용. 재평가 금지")
if LOCK.exists():
    stop("M2ENC_LEARNING_LOCK_WITHOUT_RESULT", lock=LOCK.read_text(encoding="utf-8"),
         note="이전 평가가 중단됨. 잠금 삭제 금지 — 대표 결정 필요")

EV = RECORDS[RECORDS.role == "EVAL"]
if len(EV) != 6 or set(EV.bearing) != set(D025_ONSET): stop("M2ENC_EVAL_SCOPE_MISMATCH", got=EV.bearing.tolist())

# ── ① M0 재현 게이트 ──
M0C, g_rows = {}, []
for r in EV.itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all(): stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X); del X
    F = np.stack([feat9(y) for y in Y])
    raw0 = FN_DET(F); res0 = run_detector(F)
    rms = np.sqrt((Y ** 2).mean(axis=1)); s = float(np.median(rms[:REG_MIN]))       # D-027
    Z = np.clip(Y / s, -Q_IN, Q_IN).astype(np.float32)                                # D-029
    M0C[r.bearing] = dict(n=r.n, Z=Z, s=s, clip_rate=float((np.abs(Y / s) > Q_IN).mean()),
                          d0=np.asarray(raw0["dists"], dtype=np.float64), thr0=float(raw0["thr"]),
                          nreg0=int(raw0["n_reg"]), res0=res0)
    del Y
    lead0 = (r.n - res0["onset"]) * 10 / 3600 if res0["detected"] else np.nan
    g_rows.append(dict(bearing=r.bearing, detected=res0["detected"], onset=res0["onset"], onset_ref=D025_ONSET[r.bearing],
                       reg_far=res0["reg_far"], reg_far_ref=D025_FAR[r.bearing], lead_h=lead0))
G = pd.DataFrame(g_rows)
G["onset_ok"] = G.onset == G.onset_ref
G["far_ok"] = (G.reg_far - G.reg_far_ref).abs() <= 5e-5
m0_lead, m0_far = float(G.lead_h.mean()), float(100 * G.reg_far.mean())
gate_ok = bool(G.detected.all() and G.onset_ok.all() and G.far_ok.all()
               and abs(m0_lead - 0.8356) <= 5e-4 and abs(m0_far - 0.424) <= 5e-4)
G.to_csv(EVAL_DIR / "m0_repro_gate.csv", index=False, encoding="utf-8-sig")
print("[GATE] M0 재현 (D-025 대조)\n", G.round(5).to_string(index=False))
print(f"[GATE] 평균 lead {m0_lead:.4f}h (기준 0.8356) / 평균 reg_FAR {m0_far:.4f}% (기준 0.424) → {'통과 ✅' if gate_ok else '실패 ❌'}")
if not gate_ok:
    stop("M2ENC_M0_REPRO_GATE_FAILED", note="M2 미평가 — 1회 기회 소모 안 됨. 이 출력 그대로 보내주세요")

# ── ② 잠금 ──
LOCK.write_text(json.dumps(dict(created=datetime.datetime.now().isoformat(), final_ckpt=FINAL_CK.name,
    final_sha256=FINAL_SHA, frozen_nb_sha256=nb_sha, decision="D-031 A"), ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[LOCK] 생성: {LOCK.name} — 이후 재평가 불가")

# ── ③ M2 평가 + 사전 고정 보고 항목 ──
rows = []
for b, c in M0C.items():
    n = c["n"]; t = np.arange(n) / (n - 1)
    E = embed(NET_F, c["Z"]); raw = FN_DET(E); res = run_detector(E)
    d = np.asarray(raw["dists"], dtype=np.float64); thr = float(raw["thr"]); nr = int(raw["n_reg"])
    on, det = res["onset"], res["detected"]; m0r = c["d0"] / c["thr0"]; on0 = c["res0"]["onset"]
    rows.append(dict(
        bearing=b, n=n, s=c["s"], clip_rate=c["clip_rate"],
        detected=det, onset=on, onset_frac=(on / n) if det else np.nan,
        lead_h=((n - on) * 10 / 3600) if det else np.nan, reg_far_pct=100 * res["reg_far"],
        m0_onset=on0, m0_onset_frac=on0 / n, m2_before_m0=bool(det and on < on0),
        rho_time=_sp(d[nr:], t[nr:]), rho_m0=_sp(d[nr:], c["d0"][nr:]),
        m0_ratio_at_onset=float(np.median(m0r[on:on + 5])) if det else np.nan,
        m0_ratio_reg=float(np.median(m0r[:c["nreg0"]])),
        post_exceed=float((d[on:] > thr).mean()) if det else np.nan,
        emb_min_std=float(E[:REG_MIN].std(0).min())))
R = pd.DataFrame(rows)
R.to_csv(EVAL_DIR / "learning_eval_m2.csv", index=False, encoding="utf-8-sig")

# ── ④ D-028 판정 ──
n_det, m_lead, m_far = int(R.detected.sum()), float(R.lead_h.mean()), float(R.reg_far_pct.mean())
checks = dict(detected_6=n_det >= CRITERIA_D028["detected_min"],
              mean_lead_ok=bool(m_lead >= CRITERIA_D028["mean_lead_h_min"]),
              mean_far_ok=bool(m_far <= CRITERIA_D028["mean_reg_far_pct_max"]))
passed = all(checks.values())
status = "M2ENC_V1_LEARNING_PASS_CONDITIONAL" if passed else "M2ENC_V1_LEARNING_FAIL_MEASURED"
improved = bool(passed and m_lead > BASELINE_D025["mean_lead_h"])
frozen_ok = hashlib.sha256(FROZEN_NB.read_bytes()).hexdigest() == nb_sha

SUMF.write_text(json.dumps(dict(
    status=status, decision="D-031 A (D-030 1회 평가)", criteria_d028=CRITERIA_D028, checks=checks,
    detected=n_det, mean_lead_h=round(m_lead, 4), mean_reg_far_pct=round(m_far, 4),
    improved_vs_d025=improved, baseline_d025=BASELINE_D025,
    m0_repro_gate=dict(passed=True, mean_lead_h=round(m0_lead, 4), mean_reg_far_pct=round(m0_far, 4)),
    report_items_prefixed=["onset_frac", "rho_time", "rho_m0", "m0_ratio_at_onset", "m0_ratio_reg", "m2_before_m0"],
    report_medians=dict(onset_frac=R.onset_frac.median(), rho_time=R.rho_time.median(), rho_m0=R.rho_m0.median(),
                        m0_ratio_at_onset=R.m0_ratio_at_onset.median(), m0_ratio_reg=R.m0_ratio_reg.median()),
    final_ckpt=FINAL_CK.name, final_sha256=FINAL_SHA, train_key=TRAIN_KEY, Q_in=Q_IN,
    filt_key="7e32cc3e140c", lp6000_coef_hash="b5543b5a8f89", m0_feature_fs=20480,
    frozen_notebook_unchanged=frozen_ok, unresolved_open=["#13", "#14"],
    interpretation_limit="FEMTO 조건부 결과. #14(수명 중반 경보 타당성) 미결. 현장 26.667kHz 성능으로 해석 금지",
    per_bearing=R.round(5).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 240)
print("\n[LEARNING 베어링별]\n", R[["bearing", "detected", "onset", "m0_onset", "onset_frac", "m0_onset_frac",
      "lead_h", "reg_far_pct", "m2_before_m0"]].round(4).to_string(index=False))
print("\n[사전 고정 보고 항목]\n", R[["bearing", "rho_time", "rho_m0", "m0_ratio_at_onset", "m0_ratio_reg",
      "post_exceed", "emb_min_std", "clip_rate"]].round(4).to_string(index=False))
print(f"\n[D-028] 검출 {n_det}/6 | 평균 lead {m_lead:.4f}h (≥0.752) | 평균 reg_FAR {m_far:.4f}% (≤0.924) → {checks}")
print(f"[D-025 대비] M0 {BASELINE_D025['mean_lead_h']}h vs M2 {m_lead:.4f}h → 개선 {'예' if improved else '아니오'}")
print(f"\nSTATUS: {status} | 동결 노트북 불변: {'✅' if frozen_ok else '❌'} | 저장: {EVAL_DIR}")


[GATE] M0 재현 (D-025 대조)
    bearing  detected  onset  onset_ref  reg_far  reg_far_ref  lead_h  onset_ok  far_ok
Bearing1_1      True   1462       1462  0.00333       0.0033 3.72500      True    True
Bearing1_2      True    826        826  0.00000       0.0000 0.12500      True    True
Bearing2_1      True    874        874  0.00441       0.0044 0.10278      True    True
Bearing2_2      True    487        487  0.00503       0.0050 0.86111      True    True
Bearing3_1      True    496        496  0.00781       0.0078 0.05278      True    True
Bearing3_2      True   1584       1584  0.00489       0.0049 0.14722      True    True
[GATE] 평균 lead 0.8356h (기준 0.8356) / 평균 reg_FAR 0.4244% (기준 0.424) → 통과 ✅

[LOCK] 생성: LEARNING_EVAL.lock — 이후 재평가 불가

[LEARNING 베어링별]
    bearing  detected  onset  m0_onset  onset_frac  m0_onset_frac  lead_h  reg_far_pct  m2_before_m0
Bearing1_1      True   1414      1462      0.5045         0.5216  3.8583       0.3333          True
Bearing1_2      True    571    